# Databricks Unity Catalog (UC)

本 Notebook 展示了如何将 UC 函数用作 LangChain 工具，同时支持 LangChain 和 LangGraph 的 Agent API。

请参阅 Databricks 文档 ([AWS](https://docs.databricks.com/en/sql/language-manual/sql-ref-syntax-ddl-create-sql-function.html)|[Azure](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/sql-ref-syntax-ddl-create-sql-function)|[GCP](https://docs.gcp.databricks.com/en/sql/language-manual/sql-ref-syntax-ddl-create-sql-function.html))，了解如何在 UC 中创建 SQL 或 Python 函数。请勿跳过函数和参数的注释，这对于 LLM 正确调用函数至关重要。

在本示例 Notebook 中，我们将创建一个简单的 Python 函数，该函数执行任意代码，并将其用作 LangChain 工具：

```sql
CREATE FUNCTION main.tools.python_exec (
  code STRING COMMENT 'Python code to execute. Remember to print the final result to stdout.'
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Executes Python code and returns its stdout.'
AS $$
  import sys
  from io import StringIO
  stdout = StringIO()
  sys.stdout = stdout
  exec(code)
  return stdout.getvalue()
$$
```

它在 Databricks SQL 数据仓库内的安全隔离环境中运行。

In [ ]:
%pip install --upgrade --quiet databricks-sdk langchain-community databricks-langchain langgraph mlflow

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from databricks_langchain import ChatDatabricks

llm = ChatDatabricks(endpoint="databricks-meta-llama-3-70b-instruct")

In [3]:
from databricks_langchain.uc_ai import (
    DatabricksFunctionClient,
    UCFunctionToolkit,
    set_uc_function_client,
)

client = DatabricksFunctionClient()
set_uc_function_client(client)

tools = UCFunctionToolkit(
    # Include functions as tools using their qualified names.
    # You can use "{catalog_name}.{schema_name}.*" to get all functions in a schema.
    function_names=["main.tools.python_exec"]
).tools

(可选) 要增加获取函数执行响应的重试时间，请设置环境变量 UC_TOOL_CLIENT_EXECUTION_TIMEOUT。默认重试时间值为 120 秒。
## LangGraph Agent 示例

In [ ]:
import os

os.environ["UC_TOOL_CLIENT_EXECUTION_TIMEOUT"] = "200"

## LangGraph 代理示例

LangGraph 是一个用于构建多轮代理应用程序的库。

LangGraph 是一个用于构建多轮代理应用程序的库。

```python
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage

class AgentState(TypedDict):
    input: str
    chat_history: list[tuple[str, str]]
    agent_scratchpad: list[tuple[str, str]]

@tool
def search(query: str) -> str:
    """
    Searches the internet for a given query.
    """
    # Replace with actual search implementation
    return f"Search results for {query}"

@tool
def multiply(a: int, b: int) -> int:
    """
    Multiplies two integers.
    """
    return a * b

tools = [search, multiply]

def agent(state: AgentState):
    # Replace with your agent's logic
    # This is a placeholder that just returns the last human message
    # In a real application, you would use an LLM to decide what to do
    # based on the state.
    return {"agent_scratchpad": [("tool_code", "print('hello')")]}

def route_tool(state: AgentState) -> str:
    # Replace with your tool routing logic
    # This is a placeholder that always routes to the search tool
    # In a real application, you would use an LLM to decide which tool to use
    # based on the state.
    return "search"

def call_search(state: AgentState) -> dict:
    tool_code = state["agent_scratchpad"][-1][1]
    # In a real application, you would parse the tool_code to get the tool name and arguments
    # and then call the appropriate tool.
    # For this example, we'll just call the search tool with a dummy query.
    observation = search.invoke({"query": "dummy query"})
    return {"chat_history": [("user", state["input"]), ("assistant", observation)]}

def call_multiply(state: AgentState) -> dict:
    tool_code = state["agent_scratchpad"][-1][1]
    # In a real application, you would parse the tool_code to get the tool name and arguments
    # and then call the appropriate tool.
    # For this example, we'll just call the multiply tool with dummy arguments.
    observation = multiply.invoke({"a": 5, "b": 10})
    return {"chat_history": [("user", state["input"]), ("assistant", str(observation))]}

def should_continue(state: AgentState) -> str:
    # Replace with your continuation logic
    # This is a placeholder that always returns "end"
    # In a real application, you would use an LLM to decide whether to continue
    # or end the conversation based on the state.
    return "end"

workflow = StateGraph(AgentState)

workflow.add_node("agent", agent)
workflow.add_node("call_search", call_search)
workflow.add_node("call_multiply", call_multiply)

workflow.set_entry_point("agent")

workflow.add_edge("agent", "route_tool")
workflow.add_conditional_edges(
    "route_tool",
    should_continue,
    {
        "search": "call_search",
        "multiply": "call_multiply",
        "end": END,
    },
)
workflow.add_edge("call_search", "should_continue")
workflow.add_edge("call_multiply", "should_continue")

app = workflow.compile()

# Example of how to use the compiled graph
inputs = {"input": "What is the weather in San Francisco?", "chat_history": [], "agent_scratchpad": []}
for s in app.stream(inputs):
    print(s)
```

LangGraph 是一个用于构建多轮代理应用程序的库。

```python
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage

class AgentState(TypedDict):
    input: str
    chat_history: list[tuple[str, str]]
    agent_scratchpad: list[tuple[str, str]]

@tool
def search(query: str) -> str:
    """
    Searches the internet for a given query.
    """
    # Replace with actual search implementation
    return f"Search results for {query}"

@tool
def multiply(a: int, b: int) -> int:
    """
    Multiplies two integers.
    """
    return a * b

tools = [search, multiply]

def agent(state: AgentState):
    # Replace with your agent's logic
    # This is a placeholder that just returns the last human message
    # In a real application, you would use an LLM to decide what to do
    # based on the state.
    return {"agent_scratchpad": [("tool_code", "print('hello')")]}

def route_tool(state: AgentState) -> str:
    # Replace with your tool routing logic
    # This is a placeholder that always routes to the search tool
    # In a real application, you would use an LLM to decide which tool to use
    # based on the state.
    return "search"

def call_search(state: AgentState) -> dict:
    tool_code = state["agent_scratchpad"][-1][1]
    # In a real application, you would parse the tool_code to get the tool name and arguments
    # and then call the appropriate tool.
    # For this example, we'll just call the search tool with a dummy query.
    observation = search.invoke({"query": "dummy query"})
    return {"chat_history": [("user", state["input"]), ("assistant", observation)]}

def call_multiply(state: AgentState) -> dict:
    tool_code = state["agent_scratchpad"][-1][1]
    # In a real application, you would parse the tool_code to get the tool name and arguments
    # and then call the appropriate tool.
    # For this example, we'll just call the multiply tool with dummy arguments.
    observation = multiply.invoke({"a": 5, "b": 10})
    return {"chat_history": [("user", state["input"]), ("assistant", str(observation))]}

def should_continue(state: AgentState) -> str:
    # Replace with your continuation logic
    # This is a placeholder that always returns "end"
    # In a real application, you would use an LLM to decide whether to continue
    # or end the conversation based on the state.
    return "end"

workflow = StateGraph(AgentState)

workflow.add_node("agent", agent)
workflow.add_node("call_search", call_search)
workflow.add_node("call_multiply", call_multiply)

workflow.set_entry_point("agent")

workflow.add_edge("agent", "route_tool")
workflow.add_conditional_edges(
    "route_tool",
    should_continue,
    {
        "search": "call_search",
        "multiply": "call_multiply",
        "end": END,
    },
)
workflow.add_edge("call_search", "should_continue")
workflow.add_edge("call_multiply", "should_continue")

app = workflow.compile()

# Example of how to use the compiled graph
inputs = {"input": "What is the weather in San Francisco?", "chat_history": [], "agent_scratchpad": []}
for s in app.stream(inputs):
    print(s)

In [4]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    llm,
    tools,
    prompt="You are a helpful assistant. Make sure to use tool for information.",
)
agent.invoke({"messages": [{"role": "user", "content": "36939 * 8922.4"}]})

{'messages': [HumanMessage(content='36939 * 8922.4', additional_kwargs={}, response_metadata={}, id='1a10b10b-8e37-48c7-97a1-cac5006228d5'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_a8f3986f-4b91-40a3-8d6d-39f431dab69b', 'type': 'function', 'function': {'name': 'main__tools__python_exec', 'arguments': '{"code": "print(36939 * 8922.4)"}'}}]}, response_metadata={'prompt_tokens': 771, 'completion_tokens': 29, 'total_tokens': 800}, id='run-865c3613-20ba-4e80-afc8-fde1cfb26e5a-0', tool_calls=[{'name': 'main__tools__python_exec', 'args': {'code': 'print(36939 * 8922.4)'}, 'id': 'call_a8f3986f-4b91-40a3-8d6d-39f431dab69b', 'type': 'tool_call'}]),
  ToolMessage(content='{"format": "SCALAR", "value": "329584533.59999996\\n", "truncated": false}', name='main__tools__python_exec', id='8b63d4c8-1a3d-46a5-a719-393b2ef36770', tool_call_id='call_a8f3986f-4b91-40a3-8d6d-39f431dab69b'),
  AIMessage(content='The result of the multiplication is:\n\n329584533.59999996', addit

## LangChain Agent 示例

An agent is a LangChain concept that uses a language model to decide which actions to take. These actions can be anything from calling an API, to querying a database, to searching the web. The agent then takes those actions and observes the results, and uses those results to decide what to do next.

Agent 的一个 LangChain 概念，它使用语言模型来决定采取哪些行动。这些行动可以是调用 API、查询数据库、搜索网络等任何事情。然后，Agent 会采取这些行动并观察结果，并利用这些结果来决定下一步该做什么。

This example shows how to use an agent to answer questions about a document.

本示例展示了如何使用 Agent 来回答有关文档的问题。

First, we need to install the necessary libraries:

首先，我们需要安装必要的库：

```bash
pip install langchain openai chromadb tiktoken
```

```bash
pip install langchain openai chromadb tiktoken
```

Next, we need to set up our environment variables. We'll need an OpenAI API key.

接下来，我们需要设置环境变量。我们将需要一个 OpenAI API 密钥。

```bash
export OPENAI_API_KEY="YOUR_OPENAI_API_KEY"
```

```bash
export OPENAI_API_KEY="YOUR_OPENAI_API_KEY"
```

Now, we can write the Python code to create and use the agent.

现在，我们可以编写 Python 代码来创建和使用 Agent。

```python
from langchain.agents import AgentExecutor, create_react_agent
from langchain_community.chat_models import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.llms import OpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Load the document
loader = PyPDFLoader("example.pdf")
documents = loader.load()

# Split the document into chunks and create embeddings
embeddings = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(documents, embeddings)

# Create a retriever
retriever = vectorstore.as_retriever()

# Define the prompt template
template = """Answer the question based only on the provided context. If you don't know the answer, just say that you don't know, don't try to make up an answer.

{context}

Question: {question}
Helpful Answer:"""

prompt = PromptTemplate.from_template(template)

# Create the LLM
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

# Create the RAG chain
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
)

# Define the tools the agent can use
tools = [
    Tool(
        name="RAG",
        func=rag_chain.invoke,
        description="Useful for answering questions about the document.",
    )
]

# Create the agent
agent = create_react_agent(llm, tools, prompt)

# Create the agent executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Ask a question
question = "What is the main topic of the document?"
response = agent_executor.invoke({"input": question})

print(response["output"])
```

```python
from langchain.agents import AgentExecutor, create_react_agent
from langchain_community.chat_models import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.llms import OpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain.tools import Tool # Import the Tool class

# Load the document
loader = PyPDFLoader("example.pdf")
documents = loader.load()

# Split the document into chunks and create embeddings
embeddings = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(documents, embeddings)

# Create a retriever
retriever = vectorstore.as_retriever()

# Define the prompt template
template = """Answer the question based only on the provided context. If you don't know the answer, just say that you don't know, don't try to make up an answer.

{context}

Question: {question}
Helpful Answer:"""

prompt = PromptTemplate.from_template(template)

# Create the LLM
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

# Create the RAG chain
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
)

# Define the tools the agent can use
tools = [
    Tool(
        name="RAG",
        func=rag_chain.invoke,
        description="Useful for answering questions about the document.",
    )
]

# Create the agent
agent = create_react_agent(llm, tools, prompt)

# Create the agent executor
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# Ask a question
question = "What is the main topic of the document?"
response = agent_executor.invoke({"input": question})

print(response["output"])
```

This example first loads a PDF document and creates embeddings for it. Then, it creates a retriever that can be used to retrieve relevant chunks of the document. The agent is then created with a single tool: the RAG chain. The RAG chain is responsible for answering questions about the document.

本示例首先加载一个 PDF 文档并为其创建嵌入。然后，它创建一个检索器，可用于检索文档的相关块。接着，Agent 使用一个工具创建：RAG 链。RAG 链负责回答有关文档的问题。

When the agent is asked a question, it uses the RAG chain to find the answer in the document. The agent then returns the answer to the user.

当 Agent 被问及问题时，它会使用 RAG 链在文档中查找答案。然后，Agent 将答案返回给用户。

This is a simple example, but it shows the power of agents in LangChain. Agents can be used to create more complex applications by combining multiple tools and LLMs.

这是一个简单的示例，但它展示了 LangChain 中 Agent 的强大功能。通过组合多个工具和 LLM，Agent 可用于创建更复杂的应用程序。

In [5]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Make sure to use tool for information.",
        ),
        ("placeholder", "{chat_history}"),
        ("human", "{input}"),
        ("placeholder", "{agent_scratchpad}"),
    ]
)

agent = create_tool_calling_agent(llm, tools, prompt)

In [6]:
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
agent_executor.invoke({"input": "36939 * 8922.4"})



> Entering new AgentExecutor chain...

Invoking: `main__tools__python_exec` with `{'code': 'print(36939 * 8922.4)'}`


{"format": "SCALAR", "value": "329584533.59999996\n", "truncated": false}The result of the multiplication is:

329584533.59999996

> Finished chain.


{'input': '36939 * 8922.4',
 'output': 'The result of the multiplication is:\n\n329584533.59999996'}